# 数据科学大作业 · Colab 运行入口

本笔记本用于在 **Google Colab** 上运行本项目（因果语言建模 · MHA/GQA/MQA 对比实验）。

默认场景：**冒烟测试**（opt-1.3b × wikitext，小样本，T4 可跑）。跑完整 8 组实验请换 A100，并把第 6 节里的 `smoke_config` 改成 `configs/lm_config.yaml`。

运行前准备：
1. `修改 → 笔记本设置 → 硬件加速器` 选择 **T4** 或 **A100**。
2. 左侧🔑 **Secrets** 里添加密钥 `HF_TOKEN`（HuggingFace token，需已申请 Llama-3.2-1B 访问权限）。
3. 依次执行下面每一个 Cell。

## 1. 检查 GPU 与基础环境

In [ ]:
# 查看 GPU 型号与显存，若无 GPU 这里会直接报错
!nvidia-smi

## 2. 挂载 Google Drive（结果持久化）

Colab 重启会清空本地磁盘，因此把 `results/` 软链到 Drive，跑完可随时回看。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 在 Drive 根目录下建一个固定文件夹用于存本项目产物
import os
DRIVE_ROOT = '/content/drive/MyDrive/ds_project_results'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive 结果目录：', DRIVE_ROOT)

## 3. 克隆 GitHub 仓库

公开仓库直接 clone。若之前已经 clone 过，会先删除再拉取，保证代码是最新。

In [ ]:
%cd /content
!rm -rf DS_project
!git clone --depth 1 https://github.com/Frederick2313072/DS_project.git
%cd /content/DS_project
!git log -1 --oneline

## 4. 安装依赖

Colab 默认已装 torch，按项目 `requirements.txt` 补全其余依赖。

In [ ]:
!pip install -q -r requirements.txt
# 预初始化中文字体，避免第一次画图时卡住
!python -c "from mplfonts.bin.cli import init; init()"

## 5. HuggingFace 登录（读取 Colab Secrets 的 HF_TOKEN）

Llama-3.2-1B 需要授权，先在左侧 🔑 Secrets 加一条 `HF_TOKEN` 并允许本笔记本访问。

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print('HuggingFace 登录成功')

## 6. 将 results/ 软链到 Drive

这样训练脚本写到 `./results` 的文件会直接落到 Drive。

In [ ]:
import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
# 移除仓库自带的空 results/（仓库里可能没有，这里静默处理）
!rm -rf /content/DS_project/results
!ln -s {DRIVE_ROOT} /content/DS_project/results
!ls -l /content/DS_project/results

## 7. 写入冒烟测试配置

只跑 **opt-1.3b × wikitext**，500 条训练 + 500 条评估，1 个 epoch，T4 上大约几分钟。

想跑完整 8 组实验：把下一 Cell 的 `CONFIG_PATH` 改成 `configs/lm_config.yaml` 即可。

In [ ]:
smoke_config = """# Colab 冒烟测试配置（自动生成）
seed: 42
max_seq_len: 256
num_train_epochs: 1
per_device_train_batch_size: 4
per_device_eval_batch_size: 8
gradient_accumulation_steps: 4
learning_rate: 5.0e-5
warmup_ratio: 0.05
weight_decay: 0.01
fp16: true

dataset: wikitext
max_train_samples: 500
max_eval_samples: 500

output_dir: ./results
results_csv: ./results/results_lm.csv

experiments:
  - model_name: opt-1.3b
    dataset: wikitext
"""

with open('configs/lm_config_colab_smoke.yaml', 'w') as f:
    f.write(smoke_config)

CONFIG_PATH = 'configs/lm_config_colab_smoke.yaml'
print('使用配置：', CONFIG_PATH)
!cat {CONFIG_PATH}

## 8. 运行批量实验

In [ ]:
!python src/run_all.py --config {CONFIG_PATH}

## 9. 查看 CSV 结果

In [ ]:
import pandas as pd
df = pd.read_csv('results/results_lm.csv')
df

## 10. 生成可视化图表

In [ ]:
!python src/visualize_lm.py

In [ ]:
# 把 results/ 下所有 png 图嵌入显示
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('results/**/*.png', recursive=True)):
    print(p)
    display(Image(p))

## 11. （可选）打包下载结果

Drive 已自动同步，通常无需再下载；如需本地备份可执行下面这段。

In [ ]:
from google.colab import files
!tar -czf ds_results.tar.gz -C results .
files.download('ds_results.tar.gz')